In [ ]:
# =========================
# Paths
# =========================
input_root = "D:/F25-156/ISLES22_split"
output_root = "D:/F25-156/ISLES22 Split Preprocessed"

target_shape = (96, 96, 96)

# =========================
# Transform with SAVE
# =========================
def get_save_transform(out_img_dir, out_mask_dir):
    return Compose([
        LoadImaged(keys=["image", "label"]),
        EnsureChannelFirstd(keys=["image", "label"]),
        Orientationd(keys=["image", "label"], axcodes="RAS"),
        Spacingd(
            keys=["image", "label"],
            pixdim=(1.0, 1.0, 1.0),
            mode=("bilinear", "nearest")
        ),
        CropForegroundd(keys=["image", "label"], source_key="image"),
        ResizeWithPadOrCropd(
            keys=["image", "label"],
            spatial_size=target_shape,
            mode=("constant", "constant")
        ),
        NormalizeIntensityd(keys=["image"], nonzero=True),

        # Save image
        SaveImaged(
            keys=["image"],
            output_dir=out_img_dir,
            output_postfix="",
            resample=False,
            separate_folder=False
        ),

        # Save mask
        SaveImaged(
            keys=["label"],
            output_dir=out_mask_dir,
            output_postfix="",
            resample=False,
            separate_folder=False
        ),
    ])

In [ ]:
import os
from glob import glob
from tqdm import tqdm
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Orientationd,
    Spacingd, CropForegroundd, ResizeWithPadOrCropd,
    NormalizeIntensityd, SaveImaged
)

In [ ]:
def extract_id(filepath):
    name = os.path.basename(filepath)

    # Remove known suffixes
    name = name.replace("_dwi.nii", "")
    name = name.replace("_dwi.nii.gz", "")
    name = name.replace("_lesion-msk.nii", "")
    name = name.replace("_lesion-msk.nii.gz", "")

    return name


def process_split(in_img_dir, in_mask_dir, out_img_dir, out_mask_dir):
    os.makedirs(out_img_dir, exist_ok=True)
    os.makedirs(out_mask_dir, exist_ok=True)

    dwi_files = sorted(glob(os.path.join(in_img_dir, "*.nii*")))
    mask_files = sorted(glob(os.path.join(in_mask_dir, "*.nii*")))

    assert len(dwi_files) == len(mask_files), "Mismatch in file counts!"

    transform = get_save_transform(out_img_dir, out_mask_dir)

    print(f"\n📂 Processing: {in_img_dir}")
    print(f"➡️  Input samples: {len(dwi_files)}")

    mismatch_count = 0

    for i in tqdm(range(len(dwi_files))):
        img_path = dwi_files[i]
        mask_path = mask_files[i]

        img_id = extract_id(img_path)
        mask_id = extract_id(mask_path)

        # 🔍 Check pairing
        if img_id != mask_id:
            print(f"❌ MISMATCH:")
            print(f"   Image: {os.path.basename(img_path)}")
            print(f"   Mask : {os.path.basename(mask_path)}")
            mismatch_count += 1
            continue  # skip bad pair

        # ✅ Process if correct
        data = {
            "image": img_path,
            "label": mask_path
        }

        transform(data)

    if mismatch_count > 0:
        print(f"\n⚠️ Total mismatches skipped: {mismatch_count}")
    else:
        print("✅ All pairs verified correctly!")

    # Count output
    out_imgs = len(glob(os.path.join(out_img_dir, "*.nii*")))

    print(f"📊 Side-by-side → Input: {len(dwi_files)} | Output: {out_imgs}")

In [ ]:
# =========================
# Run for ALL splits
# =========================

# ---- Clients ----
train_root = os.path.join(input_root, "train")
clients = sorted(os.listdir(train_root))

for client in clients:
    in_img = os.path.join(train_root, client, "DWI")
    in_mask = os.path.join(train_root, client, "masks")

    out_img = os.path.join(output_root, "train", client, "DWI")
    out_mask = os.path.join(output_root, "train", client, "masks")

    process_split(in_img, in_mask, out_img, out_mask)

# ---- Validation ----
process_split(
    os.path.join(input_root, "val/DWI"),
    os.path.join(input_root, "val/masks"),
    os.path.join(output_root, "val/DWI"),
    os.path.join(output_root, "val/masks"),
)

# ---- Test ----
process_split(
    os.path.join(input_root, "test/DWI"),
    os.path.join(input_root, "test/masks"),
    os.path.join(output_root, "test/DWI"),
    os.path.join(output_root, "test/masks"),
)

print("\n🎉 ALL preprocessing completed successfully!")

c:\Users\LAB.LAB12-PC14\AppData\Local\Programs\Python\Python310\lib\site-packages\monai\utils\deprecate_utils.py:321: FutureWarning: monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)



📂 Processing: D:/F25-156/ISLES24 Split\train\client_0\DWI
➡️  Input samples: 50


  0%|          | 0/50 [00:00<?, ?it/s]

2026-04-15 09:23:11,495 INFO image_writer.py:197 - writing: D:\F25-156\ISLES24 Split Preprocessed\train\client_0\DWI\sub-stroke0002_ses-02_space-ncct_dwi.nii.gz
2026-04-15 09:23:11,642 INFO image_writer.py:197 - writing: D:\F25-156\ISLES24 Split Preprocessed\train\client_0\masks\sub-stroke0002_ses-02_space-ncct_lesion-msk.nii.gz


  2%|▏         | 1/50 [00:45<36:52, 45.15s/it]

...

2026-04-15 10:40:16,102 INFO image_writer.py:197 - writing: D:\F25-156\ISLES24 Split Preprocessed\test\DWI\sub-stroke0180_ses-02_space-ncct_dwi.nii.gz
2026-04-15 10:40:16,271 INFO image_writer.py:197 - writing: D:\F25-156\ISLES24 Split Preprocessed\test\masks\sub-stroke0180_ses-02_space-ncct_lesion-msk.nii.gz


 97%|█████████▋| 28/29 [14:53<00:22, 22.13s/it]

2026-04-15 10:41:02,097 INFO image_writer.py:197 - writing: D:\F25-156\ISLES24 Split Preprocessed\test\DWI\sub-stroke0188_ses-02_space-ncct_dwi.nii.gz
2026-04-15 10:41:02,251 INFO image_writer.py:197 - writing: D:\F25-156\ISLES24 Split Preprocessed\test\masks\sub-stroke0188_ses-02_space-ncct_lesion-msk.nii.gz


100%|██████████| 29/29 [15:39<00:00, 32.39s/it]

✅ All pairs verified correctly!
📊 Side-by-side → Input: 29 | Output: 29

🎉 ALL preprocessing completed successfully!
